# Day 2 — Data Cleaning (Student Performance)

**MentorMind AI** · Week 2

Run in **Jupyter** or **Google Colab**.

Concepts: null values · duplicates · outliers · feature encoding · normalization

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Repo root (notebook lives in notebooks/)
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = REPO / "datasets" / "raw" / "student_performance"
PROCESSED = REPO / "datasets" / "processed"

sys.path.insert(0, str(REPO / "datasets" / "scripts"))
from clean_data import clean_pipeline, load_raw, encode_categoricals, rename_columns

print("Repo:", REPO)
print("Raw data:", RAW)

## 1. Load raw data

In [ ]:
df_raw = load_raw()
print(f"Rows: {len(df_raw)}, Columns: {len(df_raw.columns)}")
df_raw.head()

## 2. Null values

In [ ]:
print("Null counts per column:")
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])

df_no_null = df_raw.dropna(subset=["G1", "G2", "G3"])
print(f"\nRows after dropping grade nulls: {len(df_no_null)}")

## 3. Duplicate rows

In [ ]:
dup_count = df_no_null.duplicated().sum()
print(f"Duplicate rows: {dup_count}")
df_dedup = df_no_null.drop_duplicates()
print(f"Rows after dedup: {len(df_dedup)}")

## 4. Rename columns

In [ ]:
df_renamed = rename_columns(df_dedup)
df_renamed.columns.tolist()

## 5. Encode categoricals (0 / 1)

Example: **gender** `F` / `M` → `0` / `1`

In [ ]:
print("Before encoding (gender sample):")
print(df_renamed["gender"].value_counts())

df_encoded = encode_categoricals(df_renamed)
print("\nAfter encoding:")
print(df_encoded["gender"].value_counts())
print("\ninternet_access (no=0, yes=1):")
print(df_encoded["internet_access"].value_counts())

## 6. Full pipeline + normalize

Runs the same logic as `python datasets/scripts/clean_data.py`

In [ ]:
report = clean_pipeline()
report

In [ ]:
df_clean = pd.read_csv(PROCESSED / "student_performance_cleaned.csv")
df_norm = pd.read_csv(PROCESSED / "student_performance_normalized.csv")

print("Cleaned shape:", df_clean.shape)
print("Normalized columns (sample):", [c for c in df_norm.columns if c.endswith("_norm")][:8])
df_clean[["gender", "study_hours", "attendance_pct", "performance_pct", "at_risk"]].head(10)

## 7. Quick sanity checks

In [ ]:
print("Performance % describe:")
print(df_clean["performance_pct"].describe())
print(f"\nAt-risk rate: {df_clean['at_risk'].mean():.1%}")